In [1]:
import sys
!{sys.executable} -m pip install -q pandas==2.2.3 requests==2.32.3 beautifulsoup4==4.12.3 rapidfuzz==3.9.7 tqdm==4.66.5 duckduckgo-search==6.3.7 feedparser==6.0.11


  DEPRECATION: Building 'sgmllib3k' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'sgmllib3k'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [2]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
import hashlib
import random
import re
import time
import unicodedata

from bs4 import BeautifulSoup
try:
    from duckduckgo_search import DDGS
except Exception as exc:
    DDGS = None
    DDG_IMPORT_ERROR = exc
else:
    DDG_IMPORT_ERROR = None
import feedparser
import pandas as pd
import requests
from rapidfuzz import fuzz
from tqdm.auto import tqdm


C:\Users\Giovanny\anaconda3\envs\fluidgpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
CWD = Path.cwd().resolve()
BASE_DIR = CWD if CWD.name == "notebooks" else CWD / "notebooks"
OUTPUT_DIR = BASE_DIR / "output"
CACHE_DIR = BASE_DIR / "cache"
RAW_DIR = BASE_DIR / "raw"
REPORTS_DIR = BASE_DIR / "reports"

for directory in [BASE_DIR, OUTPUT_DIR, CACHE_DIR, RAW_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MAX_PEOPLE = 10
MAX_RESULTS_PER_QUERY = 5
ENABLE_DUCKDUCKGO = False
ENABLE_GDELT = False
SLEEP_MIN_SECONDS = 3
SLEEP_MAX_SECONDS = 7
REQUEST_TIMEOUT = 30
HEADERS = {"User-Agent": "pep-public-intel-notebook/1.0 public-osint-research contact: local-notebook"}

print(f"BASE_DIR: {BASE_DIR}")


BASE_DIR: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks


In [4]:
INPUT_CONTEXT_COLUMNS = ["estado_input", "municipio_input", "puesto_input", "dependencia_input", "periodo_inicio_input", "periodo_fin_input", "partido_input", "rfc_input", "curp_input", "empresa_relacionada_input", "alias_input"]
EVIDENCE_COLUMNS = [
    "seed_id", "nombre_persona_input", "normalized_name_input", "estado_input", "municipio_input", "puesto_input", "dependencia_input", "periodo_inicio_input", "periodo_fin_input", "partido_input", "rfc_input", "curp_input", "empresa_relacionada_input", "alias_input",
    "source", "source_type", "source_country", "source_official", "source_category", "source_reliability",
    "matched_record_name", "matched_record_normalized_name", "matched_alias", "matched_entity_type", "matched_role", "matched_position", "matched_agency", "matched_state", "matched_municipality", "matched_country", "matched_identifier", "matched_company", "matched_rfc", "matched_curp",
    "match_score_pct", "match_strength", "match_method", "matched_fields", "conflicting_fields", "requires_review", "identity_confidence_pct", "signal_type", "signal_category", "severity", "risk_layer", "risk_level_candidate", "confidence_pct",
    "evidence_title", "evidence_summary", "evidence_snippet", "evidence_keywords", "evidence_date", "evidence_url", "source_url", "search_query", "search_engine", "retrieved_at", "raw_file_path", "raw_file_sha256",
    "involvement_summary", "explanation", "limitations", "recommended_action",
]
MEDIA_EXTRA_COLUMNS = ["media_domain", "media_language", "media_country", "keyword_hits", "adverse_keyword_count", "official_domain_flag", "primary_source_flag"]
ALL_EVIDENCE_COLUMNS = EVIDENCE_COLUMNS + MEDIA_EXTRA_COLUMNS
SUMMARY_COLUMNS = ["seed_id", "nombre_persona_input", "duckduckgo_results", "gdelt_results", "total_adverse_media_candidates", "official_domain_results", "max_match_score_pct", "max_identity_confidence_pct", "requires_review", "top_evidence_url", "top_involvement_summary", "recommended_action"]
PARTICLES = {"de", "del", "la", "las", "los", "el", "y"}
ADVERSE_KEYWORDS = ["corrupcion", "lavado", "narcotrafico", "huachicol", "desvio", "enriquecimiento ilicito", "cohecho", "peculado", "empresas fachada", "crimen organizado", "sancion", "investigacion", "sentencia", "fgr", "ofac", "dea", "doj"]
OFFICIAL_DOMAINS = ["gob.mx", "ofac.treasury.gov", "treasury.gov", "justice.gov", "un.org", "scsanctions.un.org"]


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def remove_accents(value):
    if pd.isna(value):
        return ""
    return "".join(ch for ch in unicodedata.normalize("NFKD", str(value)) if not unicodedata.combining(ch))


def normalize_text(value):
    text = remove_accents(value).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_name(value, remove_particles=False):
    tokens = normalize_text(value).split()
    if remove_particles:
        tokens = [token for token in tokens if token not in PARTICLES]
    return " ".join(tokens)


def stable_hash(value):
    return hashlib.sha256(normalize_name(value).encode("utf-8")).hexdigest()[:16]


def score_name_match(left, right):
    left_norm = normalize_name(left)
    right_norm = normalize_name(right)
    if not left_norm or not right_norm:
        return 0.0
    left_tokens = [token for token in left_norm.split() if token not in PARTICLES]
    right_tokens = [token for token in right_norm.split() if token not in PARTICLES]
    if not left_tokens or not right_tokens:
        return 0.0
    left_set = set(left_tokens); right_set = set(right_tokens)
    overlap = len(left_set & right_set)
    if overlap == 0:
        return 0.0
    smaller = min(len(left_set), len(right_set)); larger = max(len(left_set), len(right_set))
    overlap_ratio = overlap / smaller if smaller else 0.0
    scores = [fuzz.WRatio(left_norm, right_norm), fuzz.token_sort_ratio(left_norm, right_norm), fuzz.ratio(left_norm, right_norm), fuzz.partial_ratio(left_norm, right_norm)]
    if smaller >= 2 and overlap_ratio >= 0.67:
        scores.append(fuzz.token_set_ratio(left_norm, right_norm))
    raw_score = float(max(scores))
    if smaller == 1 and larger >= 3:
        raw_score = min(raw_score, 59.0)
    elif smaller == 2 and larger >= 4:
        raw_score = min(raw_score, 74.0)
    if overlap_ratio < 0.50:
        raw_score = min(raw_score, 59.0)
    elif overlap_ratio < 0.67:
        raw_score = min(raw_score, 74.0)
    return round(raw_score, 2)


def classify_match(score):
    score = float(score or 0)
    if score >= 95:
        return "exact_or_very_strong_name"
    if score >= 88:
        return "strong_candidate"
    if score >= 75:
        return "medium_candidate"
    if score >= 60:
        return "weak_candidate"
    return "discard"


def empty_evidence_df():
    return pd.DataFrame(columns=ALL_EVIDENCE_COLUMNS)


def input_has_only_name(seed):
    return bool(seed.get("has_only_name", False)) or not any(str(seed.get(col, "")).strip() for col in INPUT_CONTEXT_COLUMNS)


def domain_from_url(url):
    try:
        return urlparse(str(url)).netloc.lower().replace("www.", "")
    except Exception:
        return ""


def is_official_domain(domain):
    return any(domain == d or domain.endswith("." + d) for d in OFFICIAL_DOMAINS)


def keyword_hits(text, query=""):
    normalized = normalize_text(f"{text} {query}")
    return sorted({keyword for keyword in ADVERSE_KEYWORDS if normalize_text(keyword) in normalized})


def identity_confidence(score, seed, official_domain=False, keyword_count=0):
    confidence = float(score or 0) * 0.55 + min(keyword_count, 4) * 4
    if official_domain:
        confidence += 8
    if input_has_only_name(seed):
        confidence = min(confidence, 75)
    confidence = min(confidence, 82)
    return round(max(0, min(confidence, 100)), 2)


In [5]:
def fallback_seed_dataframe():
    print("ADVERTENCIA: no existe output/00_peps_normalized.csv; se crean datos minimos de prueba para esta corrida.")
    names = ["Claudia Sheinbaum Pardo", "Andres Manuel Lopez Obrador", "Marcelo Ebrard Casaubon", "Adan Augusto Lopez Hernandez", "Ricardo Monreal Avila", "Xochitl Galvez Ruiz", "Samuel Garcia Sepulveda", "Clara Brugada Molina", "Omar Garcia Harfuch", "Luisa Maria Alcalde Lujan"]
    return pd.DataFrame([{"seed_id": stable_hash(name), "nombre_persona_input": name, "normalized_name_input": normalize_name(name), "estado_input": "", "municipio_input": "", "puesto_input": "", "dependencia_input": "", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "has_only_name": True} for name in names])

input_file = OUTPUT_DIR / "00_peps_normalized.csv"
seeds_df = pd.read_csv(input_file, encoding="utf-8-sig").fillna("") if input_file.exists() else fallback_seed_dataframe()
for col in ["seed_id", "nombre_persona_input", "normalized_name_input"] + INPUT_CONTEXT_COLUMNS + ["has_only_name"]:
    if col not in seeds_df.columns:
        seeds_df[col] = ""
seeds_df = seeds_df.head(MAX_PEOPLE).copy()
print(f"Personas a consultar en noticias/buscadores: {len(seeds_df):,}")
display(seeds_df)


Personas a consultar en noticias/buscadores: 10


,seed_id,nombre_persona_input,normalized_name_input,token_sort_name,name_tokens,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,rfc_input,curp_input,empresa_relacionada_input,alias_input,has_only_name,input_quality_score,created_at
0,064afd0cc88b3699,Claudia Sheinbaum Pardo,claudia sheinbaum pardo,claudia pardo sheinbaum,claudia|sheinbaum|pardo,Nacional,,Presidenta de Mexico,Gobierno de Mexico,2024.0,,,,,,,False,0.77,2026-05-07T07:51:29.008342+00:00
1,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,andres lopez manuel obrador,andres|manuel|lopez|obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,,,,AMLO,False,0.99,2026-05-07T07:51:29.008342+00:00
2,55c8819cc4622591,Marcelo Ebrard Casaubon,marcelo ebrard casaubon,casaubon ebrard marcelo,marcelo|ebrard|casaubon,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
3,15b1d586cc02eb0e,Adan Augusto Lopez Hernandez,adan augusto lopez hernandez,adan augusto hernandez lopez,adan|augusto|lopez|hernandez,Tabasco,,Senador / figura publica,Senado de la Republica,,,,,,,,False,0.78,2026-05-07T07:51:29.008342+00:00
4,2597a81eb6062feb,Ricardo Monreal Avila,ricardo monreal avila,avila monreal ricardo,ricardo|monreal|avila,Zacatecas,,Legislador / figura publica,Congreso de la Union,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
5,39162acdaa914138,Xochitl Galvez Ruiz,xochitl galvez ruiz,galvez ruiz xochitl,xochitl|galvez|ruiz,Nacional,,Senadora / figura publica,Senado de la Republica,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
6,0cafd5ecac30f555,Samuel Garcia Sepulveda,samuel garcia sepulveda,garcia samuel sepulveda,samuel|garcia|sepulveda,Nuevo Leon,,Gobernador,Gobierno de Nuevo Leon,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
7,cba9b1a8ead15ac3,Clara Brugada Molina,clara brugada molina,brugada clara molina,clara|brugada|molina,Ciudad de Mexico,,Jefa de Gobierno,Gobierno de la Ciudad de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
8,69573f0325299b8b,Omar Garcia Harfuch,omar garcia harfuch,garcia harfuch omar,omar|garcia|harfuch,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
9,75c154f2e7b3a23f,Luisa Maria Alcalde Lujan,luisa maria alcalde lujan,alcalde luisa lujan maria,luisa|maria|alcalde|lujan,Nacional,,Figura publica,Gobierno de Mexico,,,,,,,,False,0.78,2026-05-07T07:51:29.008342+00:00


In [6]:
def build_queries(seed):
    name = seed.get("nombre_persona_input", "")
    terms = ["corrupcion", "lavado", "narcotrafico", "enriquecimiento ilicito", "sancion", "investigacion", "FGR", "OFAC", "contratista"]
    queries = [f'"{name}" {term}' for term in terms]
    for field in ["dependencia_input", "estado_input", "puesto_input"]:
        value = str(seed.get(field, "")).strip()
        if value:
            queries.append(f'"{name}" "{value}"')
    return list(dict.fromkeys(queries))


def polite_sleep():
    seconds = random.uniform(SLEEP_MIN_SECONDS, SLEEP_MAX_SECONDS)
    time.sleep(seconds)
    return seconds


def result_to_evidence(seed, source, source_type, title, url, snippet, query, engine, language="", country=""):
    combined = f"{title} {snippet}"
    score = max(score_name_match(seed.get("nombre_persona_input", ""), combined), fuzz.partial_ratio(seed.get("normalized_name_input", ""), normalize_text(combined)))
    score = round(float(score), 2)
    if classify_match(score) == "discard":
        return None
    domain = domain_from_url(url)
    official_domain = is_official_domain(domain)
    hits = keyword_hits(combined, query)
    confidence = identity_confidence(score, seed, official_domain=official_domain, keyword_count=len(hits))
    severity = "official_strong" if official_domain else "media_candidate" if source_type == "gdelt_media_result" else "search_only"
    summary = "Resumen limitado a titulo/snippet del buscador; requiere revision de fuente primaria."
    involvement = "Resultado de busqueda menciona a la persona junto con terminos de investigacion; fuente no oficial, requiere revision."
    return {
        "seed_id": seed.get("seed_id", stable_hash(seed.get("nombre_persona_input", ""))),
        "nombre_persona_input": seed.get("nombre_persona_input", ""), "normalized_name_input": seed.get("normalized_name_input", ""),
        "estado_input": seed.get("estado_input", ""), "municipio_input": seed.get("municipio_input", ""), "puesto_input": seed.get("puesto_input", ""), "dependencia_input": seed.get("dependencia_input", ""),
        "periodo_inicio_input": seed.get("periodo_inicio_input", ""), "periodo_fin_input": seed.get("periodo_fin_input", ""), "partido_input": seed.get("partido_input", ""), "rfc_input": seed.get("rfc_input", ""), "curp_input": seed.get("curp_input", ""),
        "empresa_relacionada_input": seed.get("empresa_relacionada_input", ""), "alias_input": seed.get("alias_input", ""),
        "source": source, "source_type": source_type, "source_country": country, "source_official": False, "source_category": "adverse_media_locator", "source_reliability": "search_locator_not_primary_evidence",
        "matched_record_name": title, "matched_record_normalized_name": normalize_name(title), "matched_alias": "", "matched_entity_type": "media_result", "matched_role": "", "matched_position": "", "matched_agency": domain, "matched_state": "", "matched_municipality": "", "matched_country": country, "matched_identifier": "", "matched_company": "", "matched_rfc": "", "matched_curp": "",
        "match_score_pct": score, "match_strength": classify_match(score), "match_method": "search_result_match", "matched_fields": "nombre; keyword" if hits else "nombre", "conflicting_fields": "", "requires_review": True, "identity_confidence_pct": confidence,
        "signal_type": "adverse_media_keyword_candidate" if hits else "public_search_result_candidate", "signal_category": "adverse_media_locator", "severity": severity, "risk_layer": "media_search", "risk_level_candidate": "low_signal", "confidence_pct": confidence,
        "evidence_title": str(title)[:300], "evidence_summary": summary, "evidence_snippet": str(snippet)[:700], "evidence_keywords": "; ".join(hits), "evidence_date": "", "evidence_url": url, "source_url": url, "search_query": query, "search_engine": engine, "retrieved_at": now_utc(), "raw_file_path": "", "raw_file_sha256": "",
        "involvement_summary": involvement, "explanation": "DuckDuckGo/GDELT localizan resultados publicos; no son prueba final y requieren revisar la fuente primaria.", "limitations": "Resultado basado en titulo/snippet/metadatos. Puede haber homonimos, indexacion parcial o contexto incompleto.", "recommended_action": "verify_primary_source",
        "media_domain": domain, "media_language": language, "media_country": country, "keyword_hits": "; ".join(hits), "adverse_keyword_count": len(hits), "official_domain_flag": official_domain, "primary_source_flag": official_domain,
    }


In [7]:
duckduckgo_rows = []
benchmark_rows = []
ddg_queries = 0
start_total = time.perf_counter()

if ENABLE_DUCKDUCKGO and DDGS is not None:
    with DDGS(timeout=REQUEST_TIMEOUT) as ddgs:
        for _, seed in tqdm(seeds_df.iterrows(), total=len(seeds_df), desc="DuckDuckGo personas"):
            for query in build_queries(seed):
                q_start = time.perf_counter(); ddg_queries += 1
                try:
                    results = list(ddgs.text(query, region="mx-es", safesearch="moderate", max_results=MAX_RESULTS_PER_QUERY))
                    for item in results:
                        row = result_to_evidence(seed, "DuckDuckGo", "search_result", item.get("title", ""), item.get("href", ""), item.get("body", ""), query, "DuckDuckGo")
                        if row:
                            duckduckgo_rows.append(row)
                    status = "ok"; error = ""; note = f"results={len(results)}"
                except Exception as exc:
                    status = "error"; error = str(exc)[:300]; note = "DuckDuckGo fallo; no se insiste"
                elapsed = time.perf_counter() - q_start
                benchmark_rows.append({"notebook": "03_noticias_adverse_media", "source": "DuckDuckGo", "step": "query", "duration_seconds": round(elapsed, 6), "records_processed": 1, "records_per_second": round(1 / elapsed, 2) if elapsed else 0, "status": status, "error_message": error, "notes": note})
                polite_sleep()
elif ENABLE_DUCKDUCKGO:
    benchmark_rows.append({"notebook": "03_noticias_adverse_media", "source": "DuckDuckGo", "step": "import", "duration_seconds": 0, "records_processed": 0, "records_per_second": 0, "status": "error", "error_message": str(DDG_IMPORT_ERROR)[:300], "notes": "no se pudo importar duckduckgo_search"})
else:
    benchmark_rows.append({"notebook": "03_noticias_adverse_media", "source": "DuckDuckGo", "step": "query", "duration_seconds": 0, "records_processed": 0, "records_per_second": 0, "status": "disabled", "error_message": "", "notes": "ENABLE_DUCKDUCKGO=False"})

duckduckgo_df = pd.DataFrame(duckduckgo_rows, columns=ALL_EVIDENCE_COLUMNS) if duckduckgo_rows else empty_evidence_df()
print(f"DuckDuckGo evidencia: {len(duckduckgo_df):,}")


DuckDuckGo evidencia: 0


In [8]:
def gdelt_doc_search(query):
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {"query": query, "mode": "ArtList", "format": "json", "maxrecords": MAX_RESULTS_PER_QUERY, "sort": "HybridRel"}
    response = requests.get(url, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    if response.status_code in {401, 403}:
        return [], f"{response.status_code}; no se reintenta"
    response.raise_for_status()
    return response.json().get("articles", []), "ok"

gdelt_rows = []
gdelt_queries = 0
if ENABLE_GDELT:
    for _, seed in tqdm(seeds_df.iterrows(), total=len(seeds_df), desc="GDELT personas"):
        for query in build_queries(seed):
            q_start = time.perf_counter(); gdelt_queries += 1
            try:
                articles, note = gdelt_doc_search(query)
                for article in articles:
                    snippet = " | ".join(str(article.get(field, "")) for field in ["domain", "seendate", "language", "sourcecountry"] if article.get(field))
                    row = result_to_evidence(seed, "GDELT", "gdelt_media_result", article.get("title", ""), article.get("url", ""), snippet, query, "GDELT", language=article.get("language", ""), country=article.get("sourcecountry", ""))
                    if row:
                        row["media_domain"] = article.get("domain", row.get("media_domain", ""))
                        row["evidence_date"] = article.get("seendate", "")
                        gdelt_rows.append(row)
                status = "ok"; error = ""; note = f"{note}; results={len(articles)}"
            except Exception as exc:
                status = "error"; error = str(exc)[:300]; note = "GDELT fallo; no se insiste"
            elapsed = time.perf_counter() - q_start
            benchmark_rows.append({"notebook": "03_noticias_adverse_media", "source": "GDELT", "step": "query", "duration_seconds": round(elapsed, 6), "records_processed": 1, "records_per_second": round(1 / elapsed, 2) if elapsed else 0, "status": status, "error_message": error, "notes": note})
            polite_sleep()
else:
    benchmark_rows.append({"notebook": "03_noticias_adverse_media", "source": "GDELT", "step": "query", "duration_seconds": 0, "records_processed": 0, "records_per_second": 0, "status": "disabled", "error_message": "", "notes": "ENABLE_GDELT=False"})

gdelt_df = pd.DataFrame(gdelt_rows, columns=ALL_EVIDENCE_COLUMNS) if gdelt_rows else empty_evidence_df()
print(f"GDELT evidencia: {len(gdelt_df):,}")


GDELT evidencia: 0


In [9]:
adverse_df = pd.concat([duckduckgo_df, gdelt_df], ignore_index=True) if len(duckduckgo_df) or len(gdelt_df) else empty_evidence_df()
if not adverse_df.empty:
    adverse_df = adverse_df.drop_duplicates(subset=["seed_id", "source", "evidence_url", "search_query"], keep="first")

elapsed_total = time.perf_counter() - start_total
total_queries = ddg_queries + gdelt_queries
benchmark_rows.append({"notebook": "03_noticias_adverse_media", "source": "ALL", "step": "summary", "duration_seconds": round(elapsed_total, 6), "records_processed": total_queries, "records_per_second": round(total_queries / elapsed_total, 2) if elapsed_total else 0, "status": "ok", "error_message": "", "notes": f"personas={len(seeds_df)}; queries={total_queries}; resultados={len(adverse_df)}; resultados_minuto={round(len(adverse_df)/(elapsed_total/60), 3) if elapsed_total else 0}"})

duckduckgo_path = OUTPUT_DIR / "03_duckduckgo_evidence.csv"
gdelt_path = OUTPUT_DIR / "03_gdelt_evidence.csv"
adverse_path = OUTPUT_DIR / "03_adverse_media_evidence.csv"
benchmark_path = OUTPUT_DIR / "03_benchmark_noticias.csv"
duckduckgo_df.to_csv(duckduckgo_path, index=False, encoding="utf-8-sig")
gdelt_df.to_csv(gdelt_path, index=False, encoding="utf-8-sig")
adverse_df.to_csv(adverse_path, index=False, encoding="utf-8-sig")
pd.DataFrame(benchmark_rows).to_csv(benchmark_path, index=False, encoding="utf-8-sig")
print(f"Evidence DuckDuckGo: {duckduckgo_path}")
print(f"Evidence GDELT: {gdelt_path}")
print(f"Evidence adverse media: {adverse_path}")


Evidence DuckDuckGo: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\03_duckduckgo_evidence.csv
Evidence GDELT: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\03_gdelt_evidence.csv
Evidence adverse media: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\03_adverse_media_evidence.csv


In [10]:
summary_rows = []
for _, seed in seeds_df.iterrows():
    subset = adverse_df[adverse_df["seed_id"].astype(str) == str(seed.get("seed_id", ""))] if not adverse_df.empty else empty_evidence_df()
    duck_count = int(subset["source"].astype(str).eq("DuckDuckGo").sum()) if len(subset) else 0
    gdelt_count = int(subset["source"].astype(str).eq("GDELT").sum()) if len(subset) else 0
    official_count = int(subset["official_domain_flag"].astype(str).str.lower().isin(["true", "1"]).sum()) if len(subset) else 0
    max_match = pd.to_numeric(subset["match_score_pct"], errors="coerce").fillna(0).max() if len(subset) else 0
    max_conf = pd.to_numeric(subset["identity_confidence_pct"], errors="coerce").fillna(0).max() if len(subset) else 0
    top = None
    if len(subset):
        temp = subset.copy(); temp["_score"] = pd.to_numeric(temp["match_score_pct"], errors="coerce").fillna(0)
        top = temp.sort_values("_score", ascending=False).iloc[0]
    summary_rows.append({
        "seed_id": seed.get("seed_id", stable_hash(seed.get("nombre_persona_input", ""))),
        "nombre_persona_input": seed.get("nombre_persona_input", ""),
        "duckduckgo_results": duck_count,
        "gdelt_results": gdelt_count,
        "total_adverse_media_candidates": int(len(subset)),
        "official_domain_results": official_count,
        "max_match_score_pct": round(float(max_match), 2),
        "max_identity_confidence_pct": round(float(max_conf), 2),
        "requires_review": True if len(subset) else bool(input_has_only_name(seed)),
        "top_evidence_url": top.get("evidence_url", "") if top is not None else "",
        "top_involvement_summary": top.get("involvement_summary", "") if top is not None else "",
        "recommended_action": "verify_primary_source" if len(subset) else "no_action",
    })
summary_df = pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS)
summary_path = OUTPUT_DIR / "03_adverse_media_person_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"Summary adverse media: {summary_path}")
display(summary_df)
display(pd.DataFrame(benchmark_rows).tail(20))


Summary adverse media: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\03_adverse_media_person_summary.csv

,seed_id,nombre_persona_input,duckduckgo_results,gdelt_results,total_adverse_media_candidates,official_domain_results,max_match_score_pct,max_identity_confidence_pct,requires_review,top_evidence_url,top_involvement_summary,recommended_action
0,064afd0cc88b3699,Claudia Sheinbaum Pardo,0,0,0,0,0.0,0.0,False,,,no_action
1,910a01d167e16711,Andres Manuel Lopez Obrador,0,0,0,0,0.0,0.0,False,,,no_action
2,55c8819cc4622591,Marcelo Ebrard Casaubon,0,0,0,0,0.0,0.0,False,,,no_action
3,15b1d586cc02eb0e,Adan Augusto Lopez Hernandez,0,0,0,0,0.0,0.0,False,,,no_action
4,2597a81eb6062feb,Ricardo Monreal Avila,0,0,0,0,0.0,0.0,False,,,no_action
5,39162acdaa914138,Xochitl Galvez Ruiz,0,0,0,0,0.0,0.0,False,,,no_action
6,0cafd5ecac30f555,Samuel Garcia Sepulveda,0,0,0,0,0.0,0.0,False,,,no_action
7,cba9b1a8ead15ac3,Clara Brugada Molina,0,0,0,0,0.0,0.0,False,,,no_action
8,69573f0325299b8b,Omar Garcia Harfuch,0,0,0,0,0.0,0.0,False,,,no_action
9,75c154f2e7b3a23f,Luisa Maria Alcalde Lujan,0,0,0,0,0.0,0.0,False,,,no_action


,notebook,source,step,duration_seconds,records_processed,records_per_second,status,error_message,notes
0,03_noticias_adverse_media,DuckDuckGo,query,0.000000,0,0.0,disabled,,ENABLE_DUCKDUCKGO=False
1,03_noticias_adverse_media,GDELT,query,0.000000,0,0.0,disabled,,ENABLE_GDELT=False
2,03_noticias_adverse_media,ALL,summary,0.031629,0,0.0,ok,,personas=10; queries=0; resultados=0; resultad...


In [11]:
print("Resumen notebook 03")
print(f"1. Personas procesadas: {len(seeds_df):,}")
print(f"2. Filas de evidencia generadas: {len(adverse_df):,}")
print(f"3. Personas con hits: {adverse_df['seed_id'].nunique() if not adverse_df.empty else 0:,}")
print("4. Top 10 filas tabulares:")
display(adverse_df.head(10))
print("5. CSV generados:")
for path in [duckduckgo_path, gdelt_path, adverse_path, summary_path, benchmark_path]:
    print(f"- {path}")


Resumen notebook 03
1. Personas procesadas: 10
2. Filas de evidencia generadas: 0
3. Personas con hits: 0
4. Top 10 filas tabulares:


,seed_id,nombre_persona_input,normalized_name_input,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,...,explanation,limitations,recommended_action,media_domain,media_language,media_country,keyword_hits,adverse_keyword_count,official_domain_flag,primary_source_flag


5. CSV generados:
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\03_duckduckgo_evidence.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\03_gdelt_evidence.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\03_adverse_media_evidence.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\03_adverse_media_person_summary.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\03_benchmark_noticias.csv
